In [ ]:
# TLがかなり厳しい

import sys
import pypyjit
pypyjit.set_param("max_unroll_recursion=-1")
sys.setrecursionlimit(10**7)
input=sys.stdin.readline

n=int(input())
edge=[[] for _ in range(n)]

for i in range(n-1):
    a,b,w=map(int,input().split())
    a-=1
    b-=1
    edge[a].append((w,i,b))
    edge[b].append((w,i,a))

def eulertour(n,edge):# 頂点0を根とする
    visit=[0] # stepで訪れた頂点の番号
    depth=[0] # そのstepにおける頂点の深さ
    discovery=[0]*n # その頂点を最初に訪れた時間
    finishing=[0]*n # その頂点を最後に訪れた時間+1
    edgecost=[0] # 初めて通った辺のコストと最後にその辺を訪れた時のコストをマイナスで記録したもの
    edgeidx=[[] for _ in range(n-1)] # stepで訪れた辺の番号
    step=0
    def dfs(p,c,depthcnt):
        global step
        if len(edge[c])==1 and edge[c][-1]==p:
            return
        for w,i,nc in edge[c]:
            if p==nc:
                continue
            step+=1
            discovery[nc]=step
            depth.append(depthcnt+1)
            visit.append(nc)
            edgeidx[i].append(step)
            edgecost.append(w)
            dfs(c,nc,depthcnt+1)
            step+=1
            edgeidx[i].append(step)
            edgecost.append(-w)
            depth.append(depthcnt)
            visit.append(c)
            finishing[nc]=step
        return
    dfs(-1,0,0)
    finishing[0]=step+1
    return visit,depth,discovery,finishing,edgecost,edgeidx

def seg_func1(x,y):
    x1,x2=x
    y1,y2=y
    if x1<=y1:
        return x
    else:
        return y

def seg_func2(x,y):
    return x+y

class segtree():
    n=1
    size=1
    log=2
    d=[0]
    op=None
    e=10**15
    def __init__(self,V,OP,E):
        self.n=len(V)
        self.op=OP
        self.e=E
        self.log=(self.n-1).bit_length()
        self.size=1<<self.log
        self.d=[E for i in range(2*self.size)]
        for i in range(self.n):
            self.d[self.size+i]=V[i]
        for i in range(self.size-1,0,-1):
            self.update(i)
    def set(self,p,x):
        assert 0<=p and p<self.n
        p+=self.size
        self.d[p]=x
        for i in range(1,self.log+1):
            self.update(p>>i)
    def get(self,p):
        assert 0<=p and p<self.n
        return self.d[p+self.size]
    def prod(self,l,r):
        assert 0<=l and l<=r and r<=self.n
        sml=self.e
        smr=self.e
        l+=self.size
        r+=self.size
        while(l<r):
            if (l&1):
                sml=self.op(sml,self.d[l])
                l+=1
            if (r&1):
                smr=self.op(self.d[r-1],smr)
                r-=1
            l>>=1
            r>>=1
        return self.op(sml,smr)
    def all_prod(self):
        return self.d[1]
    def max_right(self,l,f):
        assert 0<=l and l<=self.n
        assert f(self.e)
        if l==self.n:
            return self.n
        l+=self.size
        sm=self.e
        while(1):
            while(l%2==0):
                l>>=1
            if not(f(self.op(sm,self.d[l]))):
                while(l<self.size):
                    l=2*l
                    if f(self.op(sm,self.d[l])):
                        sm=self.op(sm,self.d[l])
                        l+=1
                return l-self.size
            sm=self.op(sm,self.d[l])
            l+=1
            if (l&-l)==l:
                break
        return self.n
    def min_left(self,r,f):
        assert 0<=r and r<=self.n
        assert f(self.e)
        if r==0:
            return 0
        r+=self.size
        sm=self.e
        while(1):
            r-=1
            while(r>1 and (r%2)):
                r>>=1
            if not(f(self.op(self.d[r],sm))):
                while(r<self.size):
                    r=(2*r+1)
                    if f(self.op(self.d[r],sm)):
                        sm=self.op(self.d[r],sm)
                        r-=1
                return r+1-self.size
            sm=self.op(self.d[r],sm)
            if (r& -r)==r:
                break
        return 0
    def update(self,k):
        self.d[k]=self.op(self.d[2*k],self.d[2*k+1])
    def __str__(self):
        return str([self.get(i) for i in range(self.n)])

step=0
visit,depth,discovery,finishing,edgecost,edgeidx=eulertour(n,edge)

lca=segtree([(depth[i],visit[i]) for i in range(len(depth))],seg_func1,(10**18,10**18))
def get_lca(a,b):
    l,r=min(discovery[a],discovery[b]),max(finishing[a],finishing[b])
    _,v=lca.prod(l,r)
    return v

path=segtree(edgecost,seg_func2,0)
def get_dist(a,b):
    l,r=min(discovery[a],discovery[b]),max(finishing[a],finishing[b])
    _,v=lca.prod(l,r)
    a,b,v=discovery[a],discovery[b],discovery[v]
    return path.prod(0,a+1)+path.prod(0,b+1)-2*path.prod(0,v+1)
